# Verify hardware

In [3]:
!nvidia-smi

Thu Nov 20 12:01:40 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.94                 Driver Version: 560.94         CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 Ti   WDDM  |   00000000:01:00.0  On |                  N/A |
|  0%   31C    P8              5W /  160W |     633MiB /   8188MiB |     15%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


Looking in indexes: https://download.pytorch.org/whl/cu121
     ---------------------------------------- 0.0/2.4 GB ? eta -:--:--
     ---------------------------------------- 0.0/2.4 GB 15.3 MB/s eta 0:02:40
     ---------------------------------------- 0.0/2.4 GB 19.6 MB/s eta 0:02:05
     ---------------------------------------- 0.0/2.4 GB 22.3 MB/s eta 0:01:50
     ---------------------------------------- 0.0/2.4 GB 22.8 MB/s eta 0:01:47
     ---------------------------------------- 0.0/2.4 GB 22.8 MB/s eta 0:01:47
     ---------------------------------------- 0.0/2.4 GB 23.7 MB/s eta 0:01:43
      --------------------------------------- 0.0/2.4 GB 25.2 MB/s eta 0:01:36
      --------------------------------------- 0.0/2.4 GB 25.7 MB/s eta 0:01:34
      --------------------------------------- 0.0/2.4 GB 26.2 MB/s eta 0:01:32
      --------------------------------------- 0.1/2.4 GB 26.4 MB/s eta 0:01:31
      --------------------------------------- 0.1/2.4 GB 25.3 MB/s eta 0:01:35
 

# Test PyTorch GPU access

In [5]:
import torch
print("CUDA Available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


CUDA Available: True
GPU: NVIDIA GeForce RTX 4060 Ti


# SPO Extraction and SPO Embeddings

## SPO extraction helper

In [14]:
pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl


     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     --------------------------- ------------ 8.9/12.8 MB 46.3 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 53.6 MB/s  0:00:00
  Using cached typer-0.20.0-py3-none-any.whl.metadata (16 kB)
  Using cached langcodes-3.5.0-py3-none-any.whl.metadata (29 kB)
  Using cached language_data-1.3.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/12.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.1 MB ? eta -:--:--
    --------------------------------------- 0.3/12.1 MB ? eta -:--:--
   --- ------------------------------------ 1.0/12.1 MB 2.3 MB/s eta 0:00:05
   ------- -------------------------------- 2.4/12.1 MB 3.5 MB/s eta 0:00:03
   --------------- ------------------------ 4.7/12.1 MB 5.6 MB/s eta 0:00:0

  You can safely remove it manually.
  You can safely remove it manually.


In [1]:
# build_spo_and_spo_embeddings.py
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import spacy
from transformers import AutoTokenizer, AutoModel
import torch

DATA_DIR = "D:/multimodal_pipeline/data2"
TSV_FILES = {
    "train": os.path.join(DATA_DIR, "multimodal_train.tsv"),
    "validate": os.path.join(DATA_DIR, "multimodal_validate.tsv"),
    "test": os.path.join(DATA_DIR, "multimodal_test_public.tsv"),
}

SPO_EMB_DIR = os.path.join(DATA_DIR, "spo_embeddings")  # will contain train/, validate/, test/
os.makedirs(SPO_EMB_DIR, exist_ok=True)
for split in ["train", "validate", "test"]:
    os.makedirs(os.path.join(SPO_EMB_DIR, split), exist_ok=True)

SPO_BERT_MODEL = "bert-base-uncased"
MAX_SPO_SEQ_LEN = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading spaCy model...")
nlp = spacy.load("en_core_web_sm")

print(f"Loading BERT model for SPO embeddings: {SPO_BERT_MODEL}")
spo_tokenizer = AutoTokenizer.from_pretrained(SPO_BERT_MODEL)
spo_bert = AutoModel.from_pretrained(SPO_BERT_MODEL).to(device)
spo_bert.eval()


c:\Users\NWU\miniconda3\envs\multimodal_gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading spaCy model...


c:\Users\NWU\miniconda3\envs\multimodal_gpu\lib\site-packages\spacy\util.py:969: UserWarning: [W095] Model 'en_core_web_sm' (3.7.1) was trained with spaCy v3.7.2 and may not be 100% compatible with the current version (3.8.11). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


Loading BERT model for SPO embeddings: bert-base-uncased


c:\Users\NWU\miniconda3\envs\multimodal_gpu\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\NWU\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


KeyboardInterrupt: 

## SPO extractor (very simple dependency-based heuristic)

In [ ]:
def extract_spo_triples(text, max_triples=3):
    """
    Basic SPO extractor using spaCy dependencies.
    Returns list of (subject, predicate, object) tuples.
    """
    doc = nlp(text)
    triples = []

    for sent in doc.sents:
        subj = None
        obj = None
        verb = None

        for token in sent:
            # subject
            if "subj" in token.dep_:
                subj = token.text

            # object
            if "obj" in token.dep_:
                obj = token.text

            # verb (root or main verb)
            if token.pos_ == "VERB" and (token.dep_ == "ROOT" or token.head == token):
                verb = token.lemma_

        if subj and verb and obj:
            triples.append((subj, verb, obj))
            if len(triples) >= max_triples:
                break

    return triples


### Encode SPO triples into a single vector per post

In [ ]:
@torch.no_grad()
def encode_spo_triples(triples):
    """
    Encode a list of SPO triples as a single BERT [CLS] embedding (768-d).
    If no triples: return zeros.
    """
    if not triples:
        return np.zeros(768, dtype=np.float32)

    texts = [f"{s} [SEP] {p} [SEP] {o}" for (s, p, o) in triples]
    joined = " [SEP] ".join(texts)

    enc = spo_tokenizer(
        joined,
        truncation=True,
        padding="max_length",
        max_length=MAX_SPO_SEQ_LEN,
        return_tensors="pt"
    ).to(device)

    outputs = spo_bert(**enc)
    cls_emb = outputs.last_hidden_state[:, 0, :]  # (1, 768)
    return cls_emb.squeeze(0).cpu().numpy().astype(np.float32)


# Main SPO pipeline

In [ ]:
def process_split(split):
    tsv_path = TSV_FILES[split]
    out_dir = os.path.join(SPO_EMB_DIR, split)
    print(f"\n=== Processing SPO for {split} from {tsv_path} ===")

    df = pd.read_csv(tsv_path, sep="\t")
    print(f"Loaded {len(df)} rows")

    spo_meta_records = []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        post_id = str(row["id"])
        text = str(row.get("clean_title", row.get("title", "")))

        emb_path = os.path.join(out_dir, f"{post_id}.npy")
        if os.path.exists(emb_path):
            spo_meta_records.append({"id": post_id, "spo_emb_path": emb_path})
            continue

        triples = extract_spo_triples(text)
        emb = encode_spo_triples(triples)
        np.save(emb_path, emb)

        spo_meta_records.append({"id": post_id, "spo_emb_path": emb_path})

    meta_df = pd.DataFrame(spo_meta_records)
    meta_csv_path = os.path.join(SPO_EMB_DIR, f"{split}_spo_metadata.csv")
    meta_df.to_csv(meta_csv_path, index=False)
    print(f"SPO metadata saved to {meta_csv_path}")


#### Run for all splits

In [ ]:
if __name__ == "__main__":
    for split in ["train", "validate", "test"]:
        process_split(split)



=== Processing SPO for train from D:/data\multimodal_train.tsv ===
Loaded 564000 rows


 83%|████████▎ | 465736/564000 [21:25:02<5:46:56,  4.72it/s]  

# Graph + GNN Embeddings (GCN/GAT)

## Graph building + GNN training (sketch but runnable skeleton)

In [ ]:
# build_graph_embeddings.py
import os
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
from torch import nn
from torch_geometric.data import Data
from torch_geometric.nn import GATConv
from transformers import AutoTokenizer, AutoModel

DATA_DIR = "D:/multimodal_pipeline/data"
GRAPH_DIR = os.path.join(DATA_DIR, "graph_embeddings")
os.makedirs(GRAPH_DIR, exist_ok=True)

BERT_MODEL = "bert-base-uncased"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
bert_model = AutoModel.from_pretrained(BERT_MODEL).to(device)
bert_model.eval()

MAX_TEXT_LEN = 64
GRAPH_EMB_DIM = 256


#### Encode text to BERT node features

In [ ]:
@torch.no_grad()
def encode_text(text: str) -> np.ndarray:
    enc = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=MAX_TEXT_LEN,
        return_tensors="pt"
    ).to(device)

    out = bert_model(**enc)
    cls = out.last_hidden_state[:, 0, :]  # (1, 768)
    return cls.squeeze(0).cpu().numpy().astype(np.float32)


#### Build a simple graph for one split

In [ ]:
def build_graph_for_split(split):
    tsv_path = os.path.join(DATA_DIR, f"multimodal_{split}.tsv")
    df = pd.read_csv(tsv_path, sep="\t")
    print(f"\n=== Building graph for {split} ({len(df)} posts) ===")

    ids = df["id"].astype(str).tolist()
    id2idx = {pid: i for i, pid in enumerate(ids)}

    # Node features: BERT CLS
    x = np.zeros((len(df), 768), dtype=np.float32)
    for i, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df))):
        text = str(row.get("clean_title", row.get("title", "")))
        x[i] = encode_text(text)

    # Build edges – simple heuristics
    edges = []
    # group by subreddit
    if "subreddit" in df.columns:
        for _, sub_df in df.groupby("subreddit"):
            idxs = sub_df.index.tolist()
            for i in range(len(idxs) - 1):
                u = idxs[i]
                v = idxs[i + 1]
                edges.append((u, v))
                edges.append((v, u))

    # group by domain
    if "domain" in df.columns:
        for _, ddf in df.groupby("domain"):
            idxs = ddf.index.tolist()
            for i in range(len(idxs) - 1):
                u = idxs[i]
                v = idxs[i + 1]
                edges.append((u, v))
                edges.append((v, u))

    if not edges:
        raise RuntimeError("No edges constructed – adjust heuristics")

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

    data = Data(
        x=torch.from_numpy(x),
        edge_index=edge_index
    )

    # Save mapping
    os.makedirs(os.path.join(GRAPH_DIR, split), exist_ok=True)
    with open(os.path.join(GRAPH_DIR, f"{split}_id2idx.json"), "w") as f:
        json.dump(id2idx, f)

    return data, df, id2idx


#### Define a small GAT/GCN

In [ ]:
class GraphEncoder(nn.Module):
    def __init__(self, in_dim=768, hidden_dim=256, out_dim=256, heads=2):
        super().__init__()
        self.gat1 = GATConv(in_dim, hidden_dim, heads=heads, concat=True)
        self.gat2 = GATConv(hidden_dim * heads, out_dim, heads=1, concat=False)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(0.2)

    def forward(self, x, edge_index):
        x = self.gat1(x, edge_index)
        x = self.act(x)
        x = self.dropout(x)
        x = self.gat2(x, edge_index)
        return x  # (N, out_dim)


#### Train unsupervised (simple smoothing) and save node embeddings

In [ ]:
def train_graph_encoder(data: Data, epochs=5):
    model = GraphEncoder().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    x = data.x.to(device)
    edge_index = data.edge_index.to(device)

    model.train()
    for epoch in range(1, epochs + 1):
        opt.zero_grad()
        out = model(x, edge_index)
        # simple self-reconstruction loss (smoothness)
        loss = ((out - x[:, :out.size(1)]) ** 2).mean()
        loss.backward()
        opt.step()
        print(f"Epoch {epoch}/{epochs} - Graph loss: {loss.item():.4f}")

    model.eval()
    with torch.no_grad():
        emb = model(x, edge_index).cpu().numpy().astype(np.float32)

    return emb


## Main

In [ ]:
def process_split(split):
    data, df, id2idx = build_graph_for_split(split)
    emb = train_graph_encoder(data, epochs=5)

    split_dir = os.path.join(GRAPH_DIR, split)
    records = []
    for post_id, idx in id2idx.items():
        path = os.path.join(split_dir, f"{post_id}.npy")
        np.save(path, emb[idx])
        records.append({"id": post_id, "graph_emb_path": path})

    meta_df = pd.DataFrame(records)
    meta_df.to_csv(os.path.join(GRAPH_DIR, f"{split}_graph_metadata.csv"), index=False)
    print(f"Graph embeddings saved for {split}")

if __name__ == "__main__":
    for split in ["train", "validate", "test"]:
        process_split(split)


# Full multimodal GPU training

# Config

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from transformers import AutoTokenizer, AutoModel

from tqdm import tqdm
import time
import matplotlib.pyplot as plt


## Dataset

In [ ]:
DATA_DIR = "D:/multimodal_pipeline/data"
IMAGE_DIRS = {
    "train": os.path.join(DATA_DIR, "train_images"),
    "validate": os.path.join(DATA_DIR, "validate_images"),
    "test": os.path.join(DATA_DIR, "test_images"),
}
TSV_FILES = {
    "train": os.path.join(DATA_DIR, "multimodal_train.tsv"),
    "validate": os.path.join(DATA_DIR, "multimodal_validate.tsv"),
    "test": os.path.join(DATA_DIR, "multimodal_test_public.tsv"),
}
SPO_EMB_DIR = os.path.join(DATA_DIR, "spo_embeddings")
GRAPH_EMB_DIR = os.path.join(DATA_DIR, "graph_embeddings")

TEXT_MODEL_NAME = "bert-base-uncased"
MAX_SEQ_LEN = 64
NUM_CLASSES = 6  

BATCH_SIZE = 16
NUM_EPOCHS = 15 
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 1e-4
PATIENCE = 3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## Transforms

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


## Tokenizer / text encoder for on-the-fly text embeddings

In [ ]:
text_tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
text_bert = AutoModel.from_pretrained(TEXT_MODEL_NAME).to(device)
text_bert.eval()


# Dataset class

In [ ]:
class MultimodalFakedditDataset(Dataset):
    def __init__(self, split, label_column="6_way_label", transform=None):
        self.split = split
        self.tsv_path = TSV_FILES[split]
        self.image_root = IMAGE_DIRS[split]
        self.spo_dir = os.path.join(SPO_EMB_DIR, split)
        self.graph_dir = os.path.join(GRAPH_EMB_DIR, split)
        self.label_column = label_column
        self.transform = transform

        self.df = pd.read_csv(self.tsv_path, sep="\t")
        print(f"[{split}] Loaded {len(self.df)} rows from {self.tsv_path}")

    def __len__(self):
        return len(self.df)

    def _load_image(self, post_id):
        img_path = os.path.join(self.image_root, f"{post_id}.jpg")
        if not os.path.exists(img_path):
            # fallback: black image
            img = Image.new("RGB", (224, 224), (0, 0, 0))
        else:
            img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)
        else:
            img = transforms.ToTensor()(img)

        return img

    def _load_spo_emb(self, post_id):
        path = os.path.join(self.spo_dir, f"{post_id}.npy")
        if os.path.exists(path):
            arr = np.load(path).astype(np.float32)
        else:
            arr = np.zeros(768, dtype=np.float32)  # BERT hidden size
        return torch.from_numpy(arr)

    def _load_graph_emb(self, post_id):
        path = os.path.join(self.graph_dir, f"{post_id}.npy")
        if os.path.exists(path):
            arr = np.load(path).astype(np.float32)
        else:
            arr = np.zeros(256, dtype=np.float32)
        return torch.from_numpy(arr)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        post_id = str(row["id"])
        text = str(row.get("clean_title", row.get("title", "")))
        label = int(row[self.label_column])

        # image
        image_tensor = self._load_image(post_id)

        # text: tokenize here, encode in model (to use AMP + GPU better)
        enc = text_tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=MAX_SEQ_LEN,
            return_tensors="pt"
        )
        input_ids = enc["input_ids"].squeeze(0)      # (seq_len,)
        attention_mask = enc["attention_mask"].squeeze(0)

        # SPO and graph precomputed
        spo_emb = self._load_spo_emb(post_id)       # (768,)
        graph_emb = self._load_graph_emb(post_id)   # (256,)

        return input_ids, attention_mask, image_tensor, spo_emb, graph_emb, label


# Data loaders

In [ ]:
train_dataset = MultimodalFakedditDataset("train", transform=train_transform)
val_dataset = MultimodalFakedditDataset("validate", transform=val_test_transform)
test_dataset = MultimodalFakedditDataset("test", transform=val_test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True if device.type == "cuda" else False
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True if device.type == "cuda" else False
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True if device.type == "cuda" else False
)


## Multimodal Model (with hybrid + adaptive fusion)

In [ ]:
class MultimodalFakeNewsModel(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()

        # ---- Text encoder (BERT) ----
        self.text_bert = text_bert  # reuse global to avoid re-loading
        self.text_hidden = self.text_bert.config.hidden_size  # 768
        self.text_proj = nn.Linear(self.text_hidden, 256)

        # ---- Image encoder (ResNet50) ----
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        for p in resnet.parameters():
            p.requires_grad = False
        img_feat_dim = resnet.fc.in_features  # 2048
        resnet.fc = nn.Identity()
        self.image_encoder = resnet
        self.image_proj = nn.Linear(img_feat_dim, 256)

        # ---- SPO encoder ----
        self.spo_in_dim = 768
        self.spo_proj = nn.Sequential(
            nn.Linear(self.spo_in_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # ---- Graph encoder ----
        self.graph_in_dim = 256
        self.graph_proj = nn.Sequential(
            nn.Linear(self.graph_in_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # ---- Adaptive gates (one per modality) ----
        self.gate_text = nn.Linear(256, 1)
        self.gate_image = nn.Linear(256, 1)
        self.gate_spo = nn.Linear(256, 1)
        self.gate_graph = nn.Linear(256, 1)

        # ---- Fusion MLP (Hybrid late fusion) ----
        fused_dim = 256 * 4
        self.fusion = nn.Sequential(
            nn.Linear(fused_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # Final classifier
        self.classifier = nn.Linear(256, num_classes)

    def encode_text(self, input_ids, attention_mask):
        out = self.text_bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]        # (B, 768)
        return self.text_proj(cls)                  # (B, 256)

    def encode_image(self, images):
        feats = self.image_encoder(images)          # (B, 2048)
        return self.image_proj(feats)               # (B, 256)

    def encode_spo(self, spo_embs):
        return self.spo_proj(spo_embs)              # (B, 256)

    def encode_graph(self, graph_embs):
        return self.graph_proj(graph_embs)          # (B, 256)

    def forward(self, input_ids, attention_mask, images, spo_embs, graph_embs):
        # Encode modalities
        text_f = self.encode_text(input_ids, attention_mask)
        img_f = self.encode_image(images)
        spo_f = self.encode_spo(spo_embs)
        graph_f = self.encode_graph(graph_embs)

        # Adaptive gates
        t_gate = torch.sigmoid(self.gate_text(text_f))   # (B, 1)
        v_gate = torch.sigmoid(self.gate_image(img_f))
        s_gate = torch.sigmoid(self.gate_spo(spo_f))
        g_gate = torch.sigmoid(self.gate_graph(graph_f))

        text_f = text_f * t_gate
        img_f = img_f * v_gate
        spo_f = spo_f * s_gate
        graph_f = graph_f * g_gate

        # Fuse
        fused = torch.cat([text_f, img_f, spo_f, graph_f], dim=1)
        fused = self.fusion(fused)
        logits = self.classifier(fused)
        return logits


# Training utilities

In [ ]:
class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter = 0
        self.best_state_dict = None
        self.should_stop = False

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True


## Training / evaluation functions

In [ ]:
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for batch in tqdm(loader, desc="Train", leave=False):
        input_ids, attention_mask, images, spo_embs, graph_embs, labels = batch

        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        images = images.to(device)
        spo_embs = spo_embs.to(device)
        graph_embs = graph_embs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            logits = model(input_ids, attention_mask, images, spo_embs, graph_embs)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / total
    acc = correct / total
    return avg_loss, acc


@torch.no_grad()
def eval_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for batch in tqdm(loader, desc="Val", leave=False):
        input_ids, attention_mask, images, spo_embs, graph_embs, labels = batch

        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        images = images.to(device)
        spo_embs = spo_embs.to(device)
        graph_embs = graph_embs.to(device)
        labels = labels.to(device)

        logits = model(input_ids, attention_mask, images, spo_embs, graph_embs)
        loss = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / total
    acc = correct / total
    return avg_loss, acc


## Plot learning curves

In [ ]:
def plot_learning_curves(history):
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Loss Curves")
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_acc"], label="Train Acc")
    plt.plot(epochs, history["val_acc"], label="Val Acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Accuracy Curves")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

plot_learning_curves(history)


## Training loops and curves

In [ ]:
model = MultimodalFakeNewsModel(num_classes=NUM_CLASSES).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

early_stopper = EarlyStopping(patience=PATIENCE, min_delta=0.0)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_model_path = os.path.join(DATA_DIR, "best_multimodal_model_gpu.pth")

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\n=== Epoch {epoch}/{NUM_EPOCHS} ===")
    start = time.time()

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = eval_one_epoch(model, val_loader, criterion, device)

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    elapsed = time.time() - start
    print(f"Epoch {epoch}: "
          f"Train Loss={train_loss:.4f}, Acc={train_acc:.4f} | "
          f"Val Loss={val_loss:.4f}, Acc={val_acc:.4f} | "
          f"Time={elapsed:.1f}s")

    early_stopper.step(val_loss, model)
    if early_stopper.should_stop:
        print("Early stopping triggered.")
        break

if early_stopper.best_state_dict is not None:
    model.load_state_dict(early_stopper.best_state_dict)

torch.save(model.state_dict(), best_model_path)
print("Best multimodal model saved to:", best_model_path)


# Evaluation

In [ ]:

# LOAD BEST MODEL FOR TESTING
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

all_labels = []
all_preds = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing", leave=False):
        input_ids, attention_mask, images, spo_embs, graph_embs, labels = batch
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        images = images.to(device)
        spo_embs = spo_embs.to(device)
        graph_embs = graph_embs.to(device)
        labels = labels.to(device)

        logits = model(input_ids, attention_mask, images, spo_embs, graph_embs)
        preds = logits.argmax(dim=1)

        all_labels.extend(labels.cpu().tolist())
        all_preds.extend(preds.cpu().tolist())

# Convert to numpy
y_true = np.array(all_labels)
y_pred = np.array(all_preds)

# Classification report
target_names = [f"class_{i}" for i in range(NUM_CLASSES)]
report_dict = classification_report(y_true, y_pred, target_names=target_names, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()
print("\n--- Classification Report ---")
print(report_df)

# Save report to CSV
report_csv_path = os.path.join(DATA_DIR, "multimodal_test_classification_report.csv")
report_df.to_csv(report_csv_path)
print("Classification report saved to:", report_csv_path)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=target_names, yticklabels=target_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix — Multimodal Test Set")
plt.tight_layout()
cm_path = os.path.join(DATA_DIR, "multimodal_test_confusion_matrix.png")
plt.savefig(cm_path)
plt.show()
print("Confusion matrix saved to:", cm_path)
